# Player Number-Guided AI Highlight Video Curator
**CSCI 5922 — Mateusz Muszynski & Colin Wallace**

Tracks a target jersey number through a match video and assembles a highlight reel.

### Setup checklist (first run only)
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your match video to Google Drive
3. Add your SoccerNet credentials via **Tools → Secrets** (left panel):
   - `SOCCERNET_EMAIL` — your soccer-net.org email
   - `SOCCERNET_PASS`  — password from your NDA email
4. Run cells top-to-bottom

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell, then Runtime → Run all
# ============================================================

TARGET_JERSEY = 19
JERSEY_COLOR  = None    # set 'dark' or 'light' if needed

# Path to your match video on Google Drive (mount happens in Cell 3)
VIDEO_GDRIVE_PATH = '/content/drive/MyDrive/denver.mp4'

# Set True  → delete existing weights and retrain from scratch with real SoccerNet data
# Set False → skip training if valid weights already exist in /content/models/
FORCE_RETRAIN = True

# Video trim
TRIM_START_SEC    = 300   # 5 min — skip pre-game intro
TRIM_DURATION_SEC = 1200  # 20 min of live play

# Pipeline settings
FRAME_SKIP          = 2
CONF_THRESHOLD      = 0.25
HIGHLIGHT_THRESHOLD = 0.40

# ============================================================
import os
REPO_DIR    = '/content/csci5922'
OUTPUT_DIR  = '/content/outputs'
OUTPUT_PATH = f'{OUTPUT_DIR}/highlights_jersey{TARGET_JERSEY}.mp4'
MODELS_DIR  = f'{REPO_DIR}/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Target jersey : #{TARGET_JERSEY}  colour filter: {JERSEY_COLOR}')
print(f'FORCE_RETRAIN : {FORCE_RETRAIN}')


---
## 1 — GPU check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — Runtime → Change runtime type → T4 GPU, then re-run all')


---
## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.exists(VIDEO_GDRIVE_PATH), (
    f'Video not found at {VIDEO_GDRIVE_PATH}\n'
    'Upload your .mp4 to Google Drive and update VIDEO_GDRIVE_PATH in the Config cell.'
)
print(f'Video found : {VIDEO_GDRIVE_PATH}')
print(f'Size        : {os.path.getsize(VIDEO_GDRIVE_PATH)/1e9:.2f} GB')


---
## 3 — Clone repo & install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/mateusz-muszynski/csci5922-highlight-curator.git'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull --rebase')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')
os.system('git log --oneline -3')


In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'ultralytics>=8.2.0',
    'supervision>=0.21.0',
    'opencv-python-headless',
    'albumentations',
    'ffmpeg-python',
    'imageio[ffmpeg]',
    'PyYAML', 'tqdm', 'rich',
    'SoccerNet',
], check=True)

import torch, torchvision, ultralytics, cv2
print(f'torch {torch.__version__} | tv {torchvision.__version__} | '
      f'ultralytics {ultralytics.__version__} | cv2 {cv2.__version__}')


---
## 4 — Download SoccerNet jersey data

Uses `SOCCERNET_EMAIL` and `SOCCERNET_PASS` from **Tools → Secrets**.
Skip if data is already present — the cell checks automatically.

In [ ]:
import os, pathlib
os.chdir(REPO_DIR)

# ── Load credentials from Colab Secrets ──────────────────────────────────────
try:
    from google.colab import userdata
    SOCCERNET_EMAIL = userdata.get('SOCCERNET_EMAIL')
    SOCCERNET_PASS  = userdata.get('SOCCERNET_PASS')
    print(f'Secrets loaded: {SOCCERNET_EMAIL[:4]}****')
except Exception as e:
    raise RuntimeError(
        f'Could not load Colab Secrets ({e}).\n'
        'Go to Tools → Secrets and add SOCCERNET_EMAIL + SOCCERNET_PASS, '
        'then toggle notebook access on.'
    )

# ── Count existing images ─────────────────────────────────────────────────────
JERSEY_DIR = pathlib.Path(REPO_DIR) / 'data' / 'soccernet_jersey'

def _count_images(d):
    if not d.exists():
        return 0
    return sum(
        len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
        for p in d.iterdir() if p.is_dir()
    )

img_count = _count_images(JERSEY_DIR)

if img_count >= 1000:
    print(f'Jersey dataset already present ({img_count:,} images) — skipping download.')
else:
    print(f'Downloading SoccerNet jersey-2023 (train split) ... ({img_count} images so far)')
    os.system(
        f'python scripts/download_soccernet.py '
        f'--task jersey --data-dir data/ '
        f'--username "{SOCCERNET_EMAIL}" --password "{SOCCERNET_PASS}"'
    )

img_count   = _count_images(JERSEY_DIR)
class_count = len([p for p in JERSEY_DIR.iterdir() if p.is_dir()]) if JERSEY_DIR.exists() else 0
print(f'\nJersey dataset: {class_count} classes · {img_count:,} images')

if img_count < 500:
    print('WARNING: Very few images. Check credentials and re-run.')


---
## 5 — Train models

**Jersey CNN**: real SoccerNet crops, 50 epochs (~60–90 min on T4).
**Scorer LSTM**: synthetic clips, ~5 min.

After training, weights are saved to Google Drive so you can skip retraining next session.
Set `FORCE_RETRAIN = False` in the Config cell once weights are saved.

In [ ]:
import os, shutil, torch, pathlib, gc
os.chdir(REPO_DIR)

os.makedirs(MODELS_DIR, exist_ok=True)

jersey_weights = os.path.join(MODELS_DIR, 'jersey_cnn.pt')
scorer_weights = os.path.join(MODELS_DIR, 'scorer_lstm.pt')

# ── Optional: load saved weights from Drive to skip retraining ───────────────
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/csci5922_weights'
if not FORCE_RETRAIN and os.path.exists(DRIVE_WEIGHTS_DIR):
    for fname in ('jersey_cnn.pt', 'scorer_lstm.pt'):
        src = os.path.join(DRIVE_WEIGHTS_DIR, fname)
        dst = os.path.join(MODELS_DIR, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'Loaded from Drive: {fname}')

def _weights_valid(path, prefix='backbone.'):
    if not os.path.exists(path):
        return False
    try:
        state = torch.load(path, map_location='cpu', weights_only=True)
        keys  = list(state.keys())
        return len(keys) > 0 and any(k.startswith(prefix) for k in keys)
    except Exception as exc:
        print(f'  [WARN] {path}: {exc}')
        return False

def _real_data_available():
    d = pathlib.Path(REPO_DIR) / 'data' / 'soccernet_jersey'
    if not d.exists():
        return False
    total = sum(
        len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
        for p in d.iterdir() if p.is_dir()
    )
    return total >= 500

# ── Delete stale weights when force-retraining ────────────────────────────────
if FORCE_RETRAIN:
    for fname in ('jersey_cnn.pt', 'scorer_lstm.pt'):
        p = os.path.join(MODELS_DIR, fname)
        if os.path.exists(p):
            os.remove(p)
            print(f'Deleted stale weights: {fname}')

jersey_ok = _weights_valid(jersey_weights)
scorer_ok = _weights_valid(scorer_weights)

if not jersey_ok or not scorer_ok:
    if _real_data_available():
        img_count = sum(
            len(list(p.glob('*.jpg')) + list(p.glob('*.png')))
            for p in (pathlib.Path(REPO_DIR) / 'data' / 'soccernet_jersey').iterdir()
            if p.is_dir()
        )
        print(f'\nReal SoccerNet data found ({img_count:,} images) → --kaggle-full')
        print('  Jersey CNN : 50 epochs (~60–90 min on T4)')
        print('  Scorer LSTM: synthetic clips, ~5 min\n')

        if not jersey_ok:
            os.system('python run_training.py --kaggle-full --stages jersey --device cuda --fail-fast')
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if not scorer_ok:
            os.system('python run_training.py --kaggle --stages scorer --device cuda --fail-fast')
    else:
        raise RuntimeError('No real SoccerNet data found. Run Cell 4 first.')
else:
    print('Weights valid — skipping training. (Set FORCE_RETRAIN=True to retrain.)')

assert _weights_valid(jersey_weights), 'jersey_cnn.pt missing or wrong keys'
assert _weights_valid(scorer_weights), 'scorer_lstm.pt missing or wrong keys'

# ── Save weights to Drive for next session ────────────────────────────────────
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)
for fname in ('jersey_cnn.pt', 'scorer_lstm.pt'):
    shutil.copy(os.path.join(MODELS_DIR, fname), os.path.join(DRIVE_WEIGHTS_DIR, fname))

print(f'\n✓ Weights saved to Google Drive → {DRIVE_WEIGHTS_DIR}/')
print('  Next session: set FORCE_RETRAIN = False to load from Drive and skip retraining.')


---
## 6 — Trim gameplay clip

In [ ]:
import os
os.chdir(REPO_DIR)

CLIP_PATH = f'/content/game_clip_jersey{TARGET_JERSEY}.mp4'
if not os.path.exists(CLIP_PATH):
    print(f'Trimming {TRIM_DURATION_SEC}s clip starting at {TRIM_START_SEC}s ...')
    os.system(
        f'ffmpeg -y -ss {TRIM_START_SEC} -i "{VIDEO_GDRIVE_PATH}" -t {TRIM_DURATION_SEC} '
        f'-c copy "{CLIP_PATH}" -loglevel error'
    )
print(f'Clip: {CLIP_PATH}  ({os.path.getsize(CLIP_PATH)/1e6:.1f} MB)')
VIDEO_CLIP = CLIP_PATH


---
## 7 — Run highlight pipeline

In [ ]:
import os, sys, gc, torch
os.chdir(REPO_DIR)

for mod in list(sys.modules.keys()):
    if mod.startswith('src') or mod == 'main':
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from main import run_pipeline

result = run_pipeline(
    video_path                = VIDEO_CLIP,
    jersey_number             = TARGET_JERSEY,
    output_path               = OUTPUT_PATH,
    config_path               = os.path.join(REPO_DIR, 'config.yaml'),
    device                    = 'cuda',
    conf_override             = CONF_THRESHOLD,
    highlight_thresh_override = HIGHLIGHT_THRESHOLD,
    frame_skip                = FRAME_SKIP,
    clip_length_override      = 90,
    clip_stride_override      = 15,
    jersey_color              = JERSEY_COLOR,
    debug_video               = None,
)
print(f'\nHighlight reel → {result}')


---
## 8 — Save highlight reel to Google Drive

In [ ]:
import shutil, os

if result and os.path.exists(result):
    drive_out = f'/content/drive/MyDrive/highlights_jersey{TARGET_JERSEY}.mp4'
    shutil.copy(result, drive_out)
    print(f'Saved to Drive → {drive_out}')
else:
    print(f'No highlights found for jersey #{TARGET_JERSEY}.')
    print('Try lowering HIGHLIGHT_THRESHOLD to 0.30 in the Config cell.')


---
## 9 — Preview highlight reel

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

if result and os.path.exists(result):
    PREVIEW = f'/content/reel_preview_jersey{TARGET_JERSEY}.mp4'
    os.system(
        f'ffmpeg -y -i "{result}" -vcodec libx264 -acodec aac '
        f'-crf 26 -preset fast "{PREVIEW}" -loglevel error'
    )
    data = open(PREVIEW, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(f'''
    <h3>Highlight reel — jersey #{TARGET_JERSEY}</h3>
    <video width="960" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))
else:
    print('No highlight reel to preview.')
